<a href="https://colab.research.google.com/github/ZakariyaAlHelal/Strategic-Selection-of-Technology-Project-Portfolio-Using-Heuristic-Search/blob/main/strategic_technology_portfolio_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adaptive Heuristic Project Portfolio Selection

This notebook implements a hierarchical project portfolio selection model with:

## Decision Levels
- Level 1: Feasibility screening (Kill / Consider)
- Level 2: Primary prioritization score
- Level 3: Complementary score (secondary refinement)
- Level 4: Future capability score (secondary refinement)

## Portfolio-Level Constraints
- Budget constraint
- Staff-hours constraint
- Time horizon balance
- Initiative intent balance

## Algorithms Implemented
- Greedy baseline
- Simulated Annealing (SA)
- Adaptive Simulated Annealing (Adaptive SA)


## 1. Imports

In [ ]:
import random
import math
import time
import statistics

random.seed(42)  # Reproducibility


## 2. Project Data Input

In [ ]:
# ============================================
# 2. Project Data Input (Load from GitHub Excel)
# ============================================

import pandas as pd

GITHUB_URL = "https://raw.githubusercontent.com/ZakariyaAlHelal/strategic_technology_portfolio_selection/main/data/portfolio_dataset.xlsx"

class Project:
    def __init__(self, idx, g, p, c, f, b, h, time_cat, intent_cat):
        self.idx = idx
        self.g = g
        self.p = p
        self.c = c
        self.f = f
        self.b = b
        self.h = h
        self.time_cat = time_cat
        self.intent_cat = intent_cat


def dataframe_to_projects(df):
    projects = []
    for _, row in df.iterrows():
        projects.append(
            Project(
                int(row["idx"]),
                int(row["g"]),
                float(row["p"]),
                float(row["c"]),
                float(row["f"]),
                float(row["b"]),
                float(row["h"]),
                str(row["time_cat"]),
                str(row["intent_cat"])
            )
        )
    return projects


print("Loading project data from GitHub...")

df = pd.read_excel(GITHUB_URL, sheet_name="projects")

projects = dataframe_to_projects(df)

print("Number of candidate projects:", len(projects))
print("Eligible projects (g_i=1):", sum(1 for p in projects if p.g == 1))

Loading project data from GitHub...
Number of candidate projects: 100
Eligible projects (g_i=1): 85


## 3. Resource Constraints

In [ ]:
USE_AUTO_CONSTRAINTS = True

if USE_AUTO_CONSTRAINTS:
    eligible_projects = [p for p in projects if p.g == 1]
    B_max = 0.35 * sum(p.b for p in eligible_projects)
    H_max = 0.35 * sum(p.h for p in eligible_projects)
else:
    B_max = 5000
    H_max = 8000

print("Budget limit B_max:", B_max)
print("Hours limit  H_max:", H_max)


Budget limit B_max: 4161.15
Hours limit  H_max: 8031.799999999999


## 4. Mathematical Model Parameters

In [ ]:
EPSILON = 1e-6
LAMBDA_TIME = 1.0
LAMBDA_INTENT = 1.0


## 5. Heuristic Parameters

In [ ]:
MAX_ITER = 20000
NO_IMPROVEMENT_LIMIT = 2000

INITIAL_T = 100

# Standard SA cooling
ALPHA = 0.95

# Adaptive cooling
ALPHA_FAST = 0.90
ALPHA_SLOW = 0.98

WINDOW_SIZE = 200
RHO_MIN = 0.20
RHO_MAX = 0.40
REPAIR_THRESHOLD = 0.30


## 6. Objective and Penalty Functions

In [ ]:
def compute_time_penalty(solution, projects, target):
    counts = {"Short": 0, "Medium": 0, "Long": 0}
    for x, p in zip(solution, projects):
        if x == 1:
            counts[p.time_cat] += 1
    return sum(abs(counts[k] - target[k]) for k in counts)

def compute_intent_penalty(solution, projects, target):
    counts = {"Exploratory": 0, "Exponential": 0, "Sustaining": 0}
    for x, p in zip(solution, projects):
        if x == 1:
            counts[p.intent_cat] += 1
    return sum(abs(counts[k] - target[k]) for k in counts)

def objective(solution, projects, target_time, target_intent):
    primary = sum(p.p * x for p, x in zip(projects, solution))
    secondary = sum((p.c + p.f) * x for p, x in zip(projects, solution))
    time_pen = compute_time_penalty(solution, projects, target_time)
    intent_pen = compute_intent_penalty(solution, projects, target_intent)

    return primary + EPSILON * secondary - LAMBDA_TIME * time_pen - LAMBDA_INTENT * intent_pen


## 7. Feasibility and Repair

In [ ]:
def is_feasible(solution, projects, B_max, H_max):
    total_b = sum(p.b * x for p, x in zip(projects, solution))
    total_h = sum(p.h * x for p, x in zip(projects, solution))
    return total_b <= B_max and total_h <= H_max

def repair(solution, projects, B_max, H_max):
    # Simple repair: remove randomly selected projects until feasible
    while not is_feasible(solution, projects, B_max, H_max):
        selected = [i for i, x in enumerate(solution) if x == 1]
        if not selected:
            break
        solution[random.choice(selected)] = 0
    return solution


## 8. Greedy Baseline and Dynamic Targets

In [ ]:
def greedy(projects, B_max, H_max):
    solution = [0] * len(projects)
    sorted_indices = sorted(range(len(projects)), key=lambda i: projects[i].p, reverse=True)

    for i in sorted_indices:
        if projects[i].g == 0:
            continue
        solution[i] = 1
        if not is_feasible(solution, projects, B_max, H_max):
            solution[i] = 0

    return solution

def compute_targets(solution):
    K = sum(solution)
    target_time = {"Short": round(0.3 * K), "Medium": round(0.4 * K), "Long": round(0.3 * K)}
    target_intent = {"Exploratory": round(0.3 * K), "Exponential": round(0.4 * K), "Sustaining": round(0.3 * K)}
    return target_time, target_intent

greedy_sol = greedy(projects, B_max, H_max)
target_time, target_intent = compute_targets(greedy_sol)

print("Greedy selects K =", sum(greedy_sol), "projects")
print("Target time mix:", target_time)
print("Target intent mix:", target_intent)


Greedy selects K = 26 projects
Target time mix: {'Short': 8, 'Medium': 10, 'Long': 8}
Target intent mix: {'Exploratory': 8, 'Exponential': 10, 'Sustaining': 8}


## 9. Neighborhood Operator

In [ ]:
def generate_neighbor(solution, projects, swap_prob=0.6):
    new_solution = solution[:]

    if random.random() < swap_prob:
        # Swap move
        selected = [i for i, x in enumerate(solution) if x == 1]
        unselected = [i for i, x in enumerate(solution) if x == 0 and projects[i].g == 1]
        if selected and unselected:
            i = random.choice(selected)
            j = random.choice(unselected)
            new_solution[i] = 0
            new_solution[j] = 1
    else:
        # Flip move
        eligible = [i for i, p in enumerate(projects) if p.g == 1]
        i = random.choice(eligible)
        new_solution[i] = 1 - new_solution[i]

    return new_solution


## 10. Standard Simulated Annealing (SA)

In [ ]:
def simulated_annealing(projects, B_max, H_max, target_time, target_intent):
    current = greedy(projects, B_max, H_max)
    current_value = objective(current, projects, target_time, target_intent)

    best = current[:]
    best_value = current_value

    T = INITIAL_T
    no_improvement = 0
    start_time = time.time()

    for _ in range(MAX_ITER):
        candidate = generate_neighbor(current, projects)
        candidate = repair(candidate, projects, B_max, H_max)

        candidate_value = objective(candidate, projects, target_time, target_intent)
        delta = candidate_value - current_value

        if delta >= 0 or random.random() < math.exp(delta / T):
            current = candidate
            current_value = candidate_value

            if candidate_value > best_value:
                best = candidate[:]
                best_value = candidate_value
                no_improvement = 0
            else:
                no_improvement += 1
        else:
            no_improvement += 1

        T *= ALPHA

        if no_improvement >= NO_IMPROVEMENT_LIMIT:
            break

    runtime = time.time() - start_time
    return best, best_value, runtime


## 11. Adaptive Simulated Annealing (ASA)

In [ ]:
def adaptive_simulated_annealing(projects, B_max, H_max, target_time, target_intent):
    current = greedy(projects, B_max, H_max)
    current_value = objective(current, projects, target_time, target_intent)

    best = current[:]
    best_value = current_value

    T = INITIAL_T
    no_improvement = 0

    accept_count = 0
    repair_count = 0
    swap_prob = 0.6

    start_time = time.time()

    for iteration in range(MAX_ITER):
        candidate = generate_neighbor(current, projects, swap_prob)

        if not is_feasible(candidate, projects, B_max, H_max):
            repair_count += 1
            candidate = repair(candidate, projects, B_max, H_max)

        candidate_value = objective(candidate, projects, target_time, target_intent)
        delta = candidate_value - current_value

        if delta >= 0 or random.random() < math.exp(delta / T):
            current = candidate
            current_value = candidate_value
            accept_count += 1

            if candidate_value > best_value:
                best = candidate[:]
                best_value = candidate_value
                no_improvement = 0
            else:
                no_improvement += 1
        else:
            no_improvement += 1

        # Adaptive updates every WINDOW_SIZE iterations
        if iteration > 0 and iteration % WINDOW_SIZE == 0:
            acceptance_rate = accept_count / WINDOW_SIZE
            repair_rate = repair_count / WINDOW_SIZE

            # Adaptive cooling
            if acceptance_rate > RHO_MAX:
                T *= ALPHA_FAST
            elif acceptance_rate < RHO_MIN:
                T *= ALPHA_SLOW
            else:
                T *= ALPHA

            # Adaptive move mixing
            swap_prob = 0.8 if repair_rate > REPAIR_THRESHOLD else 0.6

            accept_count = 0
            repair_count = 0
        else:
            T *= ALPHA

        if no_improvement >= NO_IMPROVEMENT_LIMIT:
            break

    runtime = time.time() - start_time
    return best, best_value, runtime


## 12. Comparison: Greedy vs SA vs ASA

In [ ]:
def run_comparison(projects, B_max, H_max, target_time, target_intent, runs=10):
    greedy_sol = greedy(projects, B_max, H_max)
    greedy_val = objective(greedy_sol, projects, target_time, target_intent)

    print("Greedy objective:", greedy_val)
    print("--------------------------------------------------")

    sa_values, sa_times = [], []
    for _ in range(runs):
        _, val, runtime = simulated_annealing(projects, B_max, H_max, target_time, target_intent)
        sa_values.append(val)
        sa_times.append(runtime)

    ada_values, ada_times = [], []
    for _ in range(runs):
        _, val, runtime = adaptive_simulated_annealing(projects, B_max, H_max, target_time, target_intent)
        ada_values.append(val)
        ada_times.append(runtime)

    print("Standard SA:")
    print("  Avg Objective:", sum(sa_values)/runs)
    print("  Std Dev:", statistics.stdev(sa_values) if runs > 1 else 0.0)
    print("  Avg Runtime:", sum(sa_times)/runs)

    print("--------------------------------------------------")

    print("Adaptive SA:")
    print("  Avg Objective:", sum(ada_values)/runs)
    print("  Std Dev:", statistics.stdev(ada_values) if runs > 1 else 0.0)
    print("  Avg Runtime:", sum(ada_times)/runs)

    print("--------------------------------------------------")

    if sum(ada_values)/runs >= sum(sa_values)/runs:
        print("Adaptive SA outperforms or equals Standard SA ✔")
    else:
        print("Adaptive SA underperforms Standard SA ❌")

run_comparison(projects, B_max, H_max, target_time, target_intent, runs=10)


Greedy objective: 214.23028599
--------------------------------------------------
Standard SA:
  Avg Objective: 223.21529871499996
  Std Dev: 0.41876158697129456
  Avg Runtime: 1.038536548614502
--------------------------------------------------
Adaptive SA:
  Avg Objective: 222.51329549000002
  Std Dev: 1.0811625200956982
  Avg Runtime: 0.47395575046539307
--------------------------------------------------
Adaptive SA underperforms Standard SA ❌


In [ ]:
run_comparison(projects, B_max, H_max, target_time, target_intent)


Greedy objective: 214.23028599
--------------------------------------------------
Standard SA:
  Avg Objective: 223.15029823499998
  Std Dev: 0.5296964862137348
  Avg Runtime: 0.44316699504852297
--------------------------------------------------
Adaptive SA:
  Avg Objective: 223.209293583
  Std Dev: 0.6678898596406766
  Avg Runtime: 0.5099533081054688
--------------------------------------------------
Adaptive SA outperforms or equals Standard SA ✔


## 13. Final Portfolio Output (Adaptive SA)

In [ ]:
# Sorted by priority "p" (descending) for analysis

def extract_selected_projects(solution, projects):
    return [projects[i] for i, x in enumerate(solution) if x == 1]

ada_sol, ada_val, ada_runtime = adaptive_simulated_annealing(projects, B_max, H_max, target_time, target_intent)
selected_projects = sorted(extract_selected_projects(ada_sol, projects), key=lambda p: p.p, reverse=True)

print("Adaptive SA best objective (single run):", ada_val)
print("Adaptive SA runtime (single run):", ada_runtime)
print("Feasible:", is_feasible(ada_sol, projects, B_max, H_max))
print("Number of selected projects:", len(selected_projects))

total_budget = sum(p.b for p in selected_projects)
total_hours = sum(p.h for p in selected_projects)
print("Total budget used:", total_budget, "| Limit:", B_max)
print("Total hours used:", total_hours, "| Limit:", H_max)

print("\nSelected Projects:")
for p in selected_projects:
    print(f"Project {p.idx:>2} | g={p.g} | p={p.p:>3} | c={p.c:>2} | f={p.f:>2} | b={p.b:>3} | h={p.h:>3} | Time={p.time_cat:<6} | Intent={p.intent_cat}")


Adaptive SA best objective (single run): 223.51028942
Adaptive SA runtime (single run): 0.45000648498535156
Feasible: True
Number of selected projects: 27
Total budget used: 4154.0 | Limit: 4161.15
Total hours used: 7560.0 | Limit: 8031.799999999999

Selected Projects:
Project 13 | g=1 | p=9.99 | c=6.96 | f=6.08 | b=170.0 | h=296.0 | Time=Medium | Intent=Exponential
Project 21 | g=1 | p=9.93 | c=8.63 | f=5.11 | b=167.0 | h=310.0 | Time=Medium | Intent=Exploratory
Project 43 | g=1 | p=9.85 | c=9.68 | f=6.43 | b=205.0 | h=358.0 | Time=Medium | Intent=Sustaining
Project 50 | g=1 | p=9.64 | c=5.39 | f=1.45 | b=140.0 | h=232.0 | Time=Medium | Intent=Exploratory
Project 99 | g=1 | p=9.63 | c=8.32 | f=1.17 | b=159.0 | h=283.0 | Time=Long   | Intent=Sustaining
Project 57 | g=1 | p=9.54 | c=2.25 | f=4.61 | b=143.0 | h=273.0 | Time=Long   | Intent=Exploratory
Project 42 | g=1 | p=9.47 | c=4.61 | f=6.57 | b=187.0 | h=287.0 | Time=Medium | Intent=Sustaining
Project 85 | g=1 | p=9.31 | c=3.42 | f=3

In [ ]:
# Original order from project list (index order)

print("\nFinal Portfolio (Adaptive SA):")

selected_projects = extract_selected_projects(ada_sol, projects)

print("Number of selected projects:", len(selected_projects))

for p in selected_projects:
    print(f"Project {p.idx} | Priority={p.p} | Budget={p.b} | Hours={p.h} | Time={p.time_cat} | Intent={p.intent_cat}")



Final Portfolio (Adaptive SA):
Number of selected projects: 27
Project 1 | Priority=7.67 | Budget=186.0 | Hours=349.0 | Time=Long | Intent=Sustaining
Project 2 | Priority=8.49 | Budget=204.0 | Hours=264.0 | Time=Short | Intent=Exponential
Project 6 | Priority=7.62 | Budget=137.0 | Hours=253.0 | Time=Long | Intent=Exploratory
Project 13 | Priority=9.99 | Budget=170.0 | Hours=296.0 | Time=Medium | Intent=Exponential
Project 14 | Priority=8.69 | Budget=137.0 | Hours=256.0 | Time=Long | Intent=Exploratory
Project 20 | Priority=6.93 | Budget=119.0 | Hours=231.0 | Time=Long | Intent=Sustaining
Project 21 | Priority=9.93 | Budget=167.0 | Hours=310.0 | Time=Medium | Intent=Exploratory
Project 23 | Priority=6.88 | Budget=148.0 | Hours=315.0 | Time=Medium | Intent=Exponential
Project 28 | Priority=8.66 | Budget=192.0 | Hours=410.0 | Time=Medium | Intent=Exploratory
Project 37 | Priority=8.5 | Budget=94.0 | Hours=213.0 | Time=Medium | Intent=Exponential
Project 42 | Priority=9.47 | Budget=187.0 